# 07 — Scalar Flash Attention（标量版 FlashAttention）

## 背景

**FlashAttention**（Dao et al., 2022）是现代大模型推理与训练中最重要的内核之一。它通过 **IO-Aware** 的分块计算，将标准注意力的内存复杂度从 $O(N^2)$ 降至 $O(N)$，大幅提升 GPU 内存带宽利用率。

标准注意力的瓶颈在于：需要将 $N \times N$ 的注意力矩阵 $QK^T$ 写入全局内存，再读回做 Softmax——这产生了巨大内存流量。

FlashAttention 的核心思想：**将 Softmax 的三遍扫描融合进分块循环**（Online Softmax），每个 tile 只在寄存器/共享内存中流转，无需显式存储 $N \times N$ 矩阵。

本节实现**标量简化版**：用逐元素乘积 `Q * K` 代替矩阵乘 $QK^T$，保留完整的 Online Softmax + 分块结构。

## 问题定义

$$\text{QK}[b, s] = Q[b, s] \times K[b, s], \quad O[b,s] = \frac{e^{\text{QK}[b,s] - \text{MAX}[b]}}{\text{SUM}[b]} \times V[b, s]$$

等价于：`torch.softmax(Q * K, dim=1) * V`

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
B_sz, S = 256, 16384
Q = torch.randn(B_sz, S, dtype=torch.float32, device="cuda")
K = torch.randn(B_sz, S, dtype=torch.float32, device="cuda")
V = torch.randn(B_sz, S, dtype=torch.float32, device="cuda")

def ref_attn(Q, K, V):
    return torch.softmax(Q * K, dim=1) * V

O_ref = ref_attn(Q, K, V)
print(f"Q/K/V: {Q.shape} → O: {O_ref.shape}")

## Triton 实现

每个 program 处理一个 batch row，三遍扫描中 QK 计算融合进去：
- 第 1 遍：计算 `q*k`，累积行最大值
- 第 2 遍：计算 `exp(q*k - max)`，存入 O，累积行和
- 第 3 遍：读 O 做归一化，乘以 V 写回

In [ ]:
@triton.jit
def _scalar_attn_kernel(Q_ptr, K_ptr, V_ptr, O_ptr, S, BLOCK_S: tl.constexpr):
    row  = tl.program_id(0)
    base = row * S

    # 第 1 遍：求 Q*K 的行最大值
    q0 = tl.load(Q_ptr + base).to(tl.float32)
    k0 = tl.load(K_ptr + base).to(tl.float32)
    row_max = q0 * k0
    for start in range(0, S, BLOCK_S):
        cols = start + tl.arange(0, BLOCK_S)
        mask = cols < S
        q = tl.load(Q_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        k = tl.load(K_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(q * k, 0))

    # 第 2 遍：exp(QK - max)，累积行和，写入 O
    row_sum = 0.0
    for start in range(0, S, BLOCK_S):
        cols = start + tl.arange(0, BLOCK_S)
        mask = cols < S
        q = tl.load(Q_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        k = tl.load(K_ptr + base + cols, mask=mask, other=0.0).to(tl.float32)
        e = tl.exp(q * k - row_max)
        tl.store(O_ptr + base + cols, e, mask=mask)
        row_sum = row_sum + tl.sum(e, 0)

    # 第 3 遍：归一化，乘以 V
    for start in range(0, S, BLOCK_S):
        cols = start + tl.arange(0, BLOCK_S)
        mask = cols < S
        o = tl.load(O_ptr + base + cols, mask=mask)
        v = tl.load(V_ptr + base + cols, mask=mask).to(tl.float32)
        tl.store(O_ptr + base + cols, o / row_sum * v, mask=mask)

BLOCK_S_TRI = 1024

def triton_attn(Q, K, V):
    O = torch.empty_like(Q)
    _scalar_attn_kernel[(Q.shape[0],)](Q, K, V, O, Q.shape[1], BLOCK_S=BLOCK_S_TRI)
    return O

O_tri = triton_attn(Q, K, V)
ok = torch.allclose(O_ref, O_tri, atol=1e-4)
print("Triton 正确性验证:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(O_ref - O_tri)).item():.6f}")

## TileLang 实现

与 Triton 逻辑相同，但使用 `T.exp2` + `log2_e` 实现更快的指数计算，`T.reduce_max` / `T.reduce_sum` 封装归约。

In [ ]:
BB, BS = 16, 128

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_scalar_attn(Q, K, V, BLOCK_B: int, BLOCK_S: int):
    log2_e = 1.44269504
    B_, S_ = T.const("B, S")
    dtype = T.float32
    Q: T.Tensor((B_, S_), dtype)
    K: T.Tensor((B_, S_), dtype)
    V: T.Tensor((B_, S_), dtype)
    O = T.empty((B_, S_), dtype)
    with T.Kernel(B_ // BLOCK_B, threads=256) as pid_b:
        Q_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
        K_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
        V_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
        O_l = T.alloc_fragment((BLOCK_B, BLOCK_S), dtype)
        mx  = T.alloc_fragment((BLOCK_B,), dtype)
        sm  = T.alloc_fragment((BLOCK_B,), dtype)
        T.fill(mx, float("-inf"))
        for s_blk in T.Serial(S_ // BLOCK_S):
            T.copy(Q[pid_b * BLOCK_B, s_blk * BLOCK_S], Q_l)
            T.copy(K[pid_b * BLOCK_B, s_blk * BLOCK_S], K_l)
            for i, j in T.Parallel(BLOCK_B, BLOCK_S):
                O_l[i, j] = Q_l[i, j] * K_l[i, j]
            T.reduce_max(O_l, mx, dim=1, clear=False)
        T.clear(sm)
        for s_blk in T.Serial(S_ // BLOCK_S):
            T.copy(Q[pid_b * BLOCK_B, s_blk * BLOCK_S], Q_l)
            T.copy(K[pid_b * BLOCK_B, s_blk * BLOCK_S], K_l)
            for i, j in T.Parallel(BLOCK_B, BLOCK_S):
                O_l[i, j] = T.exp2(log2_e * (Q_l[i, j] * K_l[i, j] - mx[i]))
            T.reduce_sum(O_l, sm, dim=1, clear=False)
            T.copy(O_l, O[pid_b * BLOCK_B, s_blk * BLOCK_S])
        for s_blk in T.Serial(S_ // BLOCK_S):
            T.copy(O[pid_b * BLOCK_B, s_blk * BLOCK_S], O_l)
            T.copy(V[pid_b * BLOCK_B, s_blk * BLOCK_S], V_l)
            for i, j in T.Parallel(BLOCK_B, BLOCK_S):
                O_l[i, j] = O_l[i, j] / sm[i] * V_l[i, j]
            T.copy(O_l, O[pid_b * BLOCK_B, s_blk * BLOCK_S])
    return O

k = tl_scalar_attn.compile(B=B_sz, S=S, BLOCK_B=BB, BLOCK_S=BS)
O_tl = k(Q.clone(), K.clone(), V.clone())
ok = torch.allclose(O_ref, O_tl, atol=1e-4)
print("TileLang 正确性验证:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(O_ref - O_tl)).item():.6f}")

## 性能对比

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch": bench(ref_attn,    [Q, K, V]),
    "Triton":  bench(triton_attn, [Q, K, V]),
    "TileLang": bench(k,          [Q, K, V]),
}

bytes_rw = B_sz * S * 4 * 4   # read Q,K,V + write O (float32)
for name, ms in results.items():
    bw = bytes_rw / ms / 1e9
    print(f"  {name:10s}: {ms:.4f} ms  ({bw:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#C44E52"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f} ms", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("延迟 (ms)"); ax.set_title(f"Scalar Attention 延迟  (B={B_sz}, S={S})")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [bytes_rw / ms / 1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("有效带宽 (TB/s)"); ax.set_title(f"Scalar Attention 带宽利用  (B={B_sz}, S={S})")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()